# Part B1 — Ordinal personalization

**Model identity**

- Personal `p` = deployed offline ensemble score `p_python_art_cfg` (weights 0.6 / 0.4). Not a LOSO fold prediction.
- Chronological split: first 80% by parsed timestamp (64/17 when n=81).
- Walk-forward curves: test set changes with k. Rolling-origin is the interpreted summary.

## Setup

In [ ]:
!pip -q install xgboost scikit-learn pandas numpy matplotlib seaborn scipy

In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = Path('/content/drive/MyDrive/cloudnine-stress-ml')
    USER_CSV = PROJECT / 'data' / 'cloudNine-database-data.csv'
    P_CSV = PROJECT / 'data' / 'phone_vs_python_p.csv'
    OUTPUT_DIR = PROJECT / 'personalization_results'
except Exception:
    PROJECT = Path.cwd()
    if not (PROJECT / 'docs').exists() and (PROJECT.parent / 'docs').exists():
        PROJECT = PROJECT.parent
    ART = PROJECT / 'docs' / 'Cloudnine Stress Artifacts'
    USER_CSV = ART / 'measurement_samples.csv'
    P_CSV = ART / 'phone_vs_python_p.csv'
    OUTPUT_DIR = ART / 'revision'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('USER_CSV', USER_CSV.exists())
print('P_CSV', P_CSV.exists())
print('OUTPUT_DIR', OUTPUT_DIR)

In [ ]:
import ast
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error, r2_score

USER_FEATURES = ['hrrange', 'hrvar', 'hrstd', 'hrmin', 'hrmax', 'hrkurt']
RANDOM_SEED = 123
K_VALUES = [3, 5, 10, 15, 20, 25, 30, 40, 50, 60]
MIN_TEST_SIZE = 5
HOLDOUT_TRAIN_FRAC = 0.8

np.random.seed(RANDOM_SEED)
sns.set_theme(style='whitegrid', context='notebook')

## Load

In [ ]:
def load_user_data(csv_path, p_csv=None):
    raw = pd.read_csv(csv_path)
    df = raw.copy()
    df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce', utc=True)
    elif 'measurement_started_at' in df.columns:
        df['timestamp'] = pd.to_datetime(df['measurement_started_at'], errors='coerce', utc=True)
    else:
        raise ValueError('need timestamp')

    if 'is_test' in df.columns:
        df = df[~df['is_test'].astype(str).str.lower().isin(['true', '1', 'yes'])]
    if 'user_rating_status' in df.columns:
        df = df[df['user_rating_status'].astype(str).str.lower() == 'submitted']

    df['y10'] = pd.to_numeric(df['user_prediction'], errors='coerce')
    df = df[df['y10'].between(1, 10)].copy()
    if 'hr_sample_count' in df.columns:
        df = df[pd.to_numeric(df['hr_sample_count'], errors='coerce').fillna(0) >= 10]

    if p_csv is not None and Path(p_csv).exists() and 'id' in df.columns:
        pdf = pd.read_csv(p_csv)
        pdf.columns = [c.strip().lower() for c in pdf.columns]
        df['id'] = df['id'].astype(str)
        pdf['id'] = pdf['id'].astype(str)
        df = df.merge(pdf[['id', 'p_python_art_cfg']], on='id', how='left')
        df['p'] = df['p_python_art_cfg'].astype(float)
        print('p source: p_python_art_cfg (deployed 0.6/0.4)')
    else:
        raise RuntimeError('offline p_python_art_cfg required')

    if df['p'].isna().any():
        raise RuntimeError(f"missing p for {df['p'].isna().sum()} rows")

    df = df.sort_values('timestamp').reset_index(drop=True)
    df['hr_mid'] = (
        pd.to_numeric(df['hrmin'], errors='coerce') + pd.to_numeric(df['hrmax'], errors='coerce')
    ) / 2.0
    act = (
        df['activity_type'].astype(str).str.lower().str.strip()
        if 'activity_type' in df.columns
        else pd.Series(['unknown'] * len(df))
    )
    df['is_rest'] = act == 'rest'
    df['is_work'] = act == 'work'
    df['activity_type_norm'] = act
    print('n=', len(df), 'rest', int(df.is_rest.sum()), 'work', int(df.is_work.sum()))
    return df, raw


user_df, raw_export = load_user_data(USER_CSV, P_CSV)
assert len(user_df) == 81
split = int(len(user_df) * HOLDOUT_TRAIN_FRAC)
assert split == 64 and len(user_df) - split == 17
user_df.head()

## Collection flow

In [ ]:
raw_n = len(raw_export)
raw_cols = {c.strip().lower().replace(' ', '_'): c for c in raw_export.columns}
is_test = raw_export[raw_cols.get('is_test', 'is_test')].astype(str).str.lower().isin(['true', '1', 'yes'])
n_test = int(is_test.sum())
ts = pd.to_datetime(user_df['timestamp'], errors='coerce', utc=True)
n_days = int(ts.dt.date.nunique()) if ts.notna().any() else None
flow = pd.DataFrame([
    {'stage': 'raw_export_rows', 'n': raw_n, 'note': 'measurement_samples.csv'},
    {'stage': 'test_rows', 'n': n_test, 'note': 'is_test'},
    {'stage': 'clean_analysis_set', 'n': len(user_df), 'note': 'submitted, y in 1-10, hr_sample_count>=10 slots'},
    {'stage': 'collection_days', 'n': n_days, 'note': f'~{len(user_df)/n_days:.2f} ratings/day' if n_days else ''},
    {'stage': 'notifications_delivered_or_tapped', 'n': None, 'note': 'not_logged'},
    {'stage': 'manual_vs_notification_start', 'n': None, 'note': 'not_logged'},
    {'stage': 'skipped_ratings', 'n': None, 'note': 'not_logged'},
])
flow.to_csv(OUTPUT_DIR / 'collection_flow.csv', index=False)
display(flow)

label_notes = {
    'exact_question': 'How stressed do you feel right now?',
    'scale_1': 'very low',
    'scale_10': 'very high',
    'time_reference': 'right now',
    'slider_default': 5,
    'n_at_slider_default_5': int((user_df['y10'] == 5).sum()),
    'n_pos_threshold_4_5': int((user_df['y10'] >= 5).sum()),
}
json.dump(label_notes, open(OUTPUT_DIR / 'label_construct_notes.json', 'w'), indent=2)
print(json.dumps(label_notes, indent=2))

## Sensor audit

In [ ]:
def parse_hr_values(cell):
    if isinstance(cell, list):
        return [float(x) for x in cell]
    if pd.isna(cell):
        return []
    s = str(cell).strip()
    try:
        v = json.loads(s)
    except Exception:
        v = ast.literal_eval(s)
    return [float(x) for x in v]


def run_length_encode(values):
    if not values:
        return [], []
    runs, lengths = [], []
    cur, n = values[0], 1
    for x in values[1:]:
        if x == cur:
            n += 1
        else:
            runs.append(cur); lengths.append(n)
            cur, n = x, 1
    runs.append(cur); lengths.append(n)
    return runs, lengths


def sample_variance(values):
    values = np.asarray(values, float)
    if len(values) < 2:
        return 0.0
    return float(np.var(values, ddof=1))


def excess_kurtosis(values):
    values = np.asarray(values, float)
    n = len(values)
    if n < 2:
        return 0.0
    mean = values.mean()
    diffs = values - mean
    m2 = np.mean(diffs ** 2)
    if m2 <= 1e-12:
        return 0.0
    return float(np.mean(diffs ** 4) / (m2 * m2) - 3.0)


audit_rows = []
for _, r in user_df.iterrows():
    if 'hr_values' not in user_df.columns:
        break
    vals = parse_hr_values(r['hr_values'])
    runs, lengths = run_length_encode(vals)
    audit_rows.append({
        'id': r.get('id'),
        'stored_len': len(vals),
        'n_runs': len(runs),
        'max_run_len': max(lengths) if lengths else 0,
        'var_stored': sample_variance(vals),
        'kurt_stored': excess_kurtosis(vals),
        'var_collapsed': sample_variance(runs),
        'kurt_collapsed': excess_kurtosis(runs),
        'y10': float(r['y10']),
        'activity_type': str(r.get('activity_type_norm', '')),
    })

audit = pd.DataFrame(audit_rows)
audit.to_csv(OUTPUT_DIR / 'sensor_runs_per_window.csv', index=False)
feat_cmp = pd.DataFrame([
    {
        'feature': 'variance',
        'mean_abs_diff': float((audit.var_collapsed - audit.var_stored).abs().mean()),
        'frac_changed': float(((audit.var_collapsed - audit.var_stored).abs() > 1e-12).mean()),
    },
    {
        'feature': 'kurtosis',
        'mean_abs_diff': float((audit.kurt_collapsed - audit.kurt_stored).abs().mean()),
        'frac_changed': float(((audit.kurt_collapsed - audit.kurt_stored).abs() > 1e-12).mean()),
    },
])
feat_cmp.to_csv(OUTPUT_DIR / 'sensor_feature_raw_vs_collapsed.csv', index=False)

summary = {
    'n_windows': int(len(audit)),
    'n_runs_median': float(audit.n_runs.median()),
    'windows_n_runs_lt_10': int((audit.n_runs < 10).sum()),
    'windows_stored_len_eq_30': int((audit.stored_len == 30).sum()),
    'dedup_healthkit_wcsession': 'not_performed_no_sample_uuid_in_export',
    'gte10_rule': 'counted_filled_1s_slots_not_unique_readings; removed_no_rows_from_this_export',
}
json.dump(summary, open(OUTPUT_DIR / 'sensor_audit_summary.json', 'w'), indent=2)
print(json.dumps(summary, indent=2))
print('stored_len==30:', int((audit.stored_len == 30).sum()), 'of', len(audit))
display(feat_cmp)

fig, ax = plt.subplots(figsize=(6, 3.5))
counts = audit.n_runs.value_counts().sort_index()
ax.bar(counts.index, counts.values, edgecolor='k', linewidth=0.3)
ax.axvline(10, color='red', linestyle='--', label='>=10 slot gate')
ax.set_xlabel('Run count')
ax.set_ylabel('Windows')
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'fig_sensor_runs_hist.png', dpi=150)
plt.show()

## Helpers

In [ ]:
def metrics_triple(y_true, y_pred):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    out = {
        'r2': float(r2_score(y_true, y_pred)) if len(y_true) >= 2 else np.nan,
        'mae': float(mean_absolute_error(y_true, y_pred)),
        'spearman': np.nan,
    }
    if len(np.unique(y_true)) >= 2 and len(np.unique(y_pred)) >= 2:
        out['spearman'] = float(spearmanr(y_true, y_pred).correlation)
    return out


def m0_predict(p):
    return np.clip(np.round(np.asarray(p, float) * 10.0), 1, 10)


def fit_m1(p_train, y_train):
    A = np.column_stack([np.asarray(p_train, float), np.ones(len(p_train))])
    a, b = np.linalg.lstsq(A, np.asarray(y_train, float), rcond=None)[0]
    return float(a), float(b)


def predict_m1(a, b, p):
    return np.clip(a * np.asarray(p, float) + b, 1, 10)


def evaluate_split(train, test):
    y_tr, y_te = train['y10'].values, test['y10'].values
    p_te = test['p'].values
    out = {}
    out['M0'] = {'pred': m0_predict(p_te), **metrics_triple(y_te, m0_predict(p_te))}
    mu, med = float(np.mean(y_tr)), float(np.median(y_tr))
    out['Naive_mean'] = {'pred': np.full(len(y_te), mu), **metrics_triple(y_te, np.full(len(y_te), mu))}
    out['Naive_median'] = {'pred': np.full(len(y_te), med), **metrics_triple(y_te, np.full(len(y_te), med))}
    if len(train) >= 3:
        a, b = fit_m1(train['p'].values, y_tr)
        pred = predict_m1(a, b, p_te)
        out['M1'] = {'pred': pred, **metrics_triple(y_te, pred), 'params': {'a': a, 'b': b}}
    return out

## In-sample baselines

In [ ]:
y = user_df['y10'].values
p = user_df['p'].values
table_ins = pd.DataFrame([
    {'method': 'M0', **metrics_triple(y, m0_predict(p)), 'n': len(y), 'note': 'in_sample'},
    {'method': 'Naive_mean', **metrics_triple(y, np.full(len(y), float(np.mean(y)))), 'n': len(y), 'note': 'in_sample'},
    {'method': 'Naive_median', **metrics_triple(y, np.full(len(y), float(np.median(y)))), 'n': len(y), 'note': 'in_sample'},
])
display(table_ins)
table_ins.to_csv(OUTPUT_DIR / 'table7_1_baselines_insample.csv', index=False)

fig, ax = plt.subplots(figsize=(4.2, 4.2))
ax.scatter(y, m0_predict(p), alpha=0.75, edgecolor='k', linewidth=0.3, s=28)
ax.plot([1, 10], [1, 10], 'k--', alpha=0.5)
ax.set_xlim(0.5, 10.5); ax.set_ylim(0.5, 10.5)
ax.set_xlabel('Self-report'); ax.set_ylabel('M0')
ax.set_aspect('equal')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'fig_m0_scatter.png', dpi=150)
plt.show()

## Shared chronological holdout + activity slices

In [ ]:
n = len(user_df)
split = int(n * HOLDOUT_TRAIN_FRAC)
train = user_df.iloc[:split].copy()
test = user_df.iloc[split:].copy().reset_index(drop=True)
y_tr, y_te = train['y10'].values, test['y10'].values
p_te = test['p'].values

pred_m0 = m0_predict(p_te)
mu, med = float(np.mean(y_tr)), float(np.median(y_tr))
pred_mean = np.full(len(y_te), mu)
pred_median = np.full(len(y_te), med)
prev = user_df['y10'].values[split - 1:n - 1]
a, b = fit_m1(train['p'].values, y_tr)
pred_m1 = predict_m1(a, b, p_te)

preds = {
    'M0': pred_m0,
    'Naive_mean': pred_mean,
    'Naive_median': pred_median,
    'Previous_rating': prev,
    'M1': pred_m1,
}

rest_mask = test['is_rest'].values
work_mask = test['is_work'].values
assert rest_mask.sum() + work_mask.sum() == len(test)

rows, raw_mae = [], {}
for subset, mask in [('overall', np.ones(len(test), bool)), ('rest', rest_mask), ('work', work_mask)]:
    for method, pred in preds.items():
        m = metrics_triple(y_te[mask], pred[mask])
        raw_mae[(subset, method)] = m['mae']
        rows.append({
            'subset': subset, 'method': method,
            'r2': m['r2'], 'mae': m['mae'], 'spearman': m['spearman'],
            'n_test': int(mask.sum()), 'n_train': len(train),
            'note': 'shared_chronological_holdout' if subset == 'overall' else 'descriptive_activity_slice',
        })

table = pd.DataFrame(rows)
for method in preds:
    o = raw_mae[('overall', method)]
    r = raw_mae[('rest', method)]
    w = raw_mae[('work', method)]
    weighted = (int(rest_mask.sum()) * r + int(work_mask.sum()) * w) / len(test)
    assert abs(weighted - o) < 1e-12, (method, weighted, o)
    print(f'ASSERT MAE weighted==overall [{method}] OK')

table.to_csv(OUTPUT_DIR / 'table7_3_activity_shared_split.csv', index=False)
json.dump({
    'n': n, 'n_train': len(train), 'n_test': len(test),
    'n_rest_test': int(rest_mask.sum()), 'n_work_test': int(work_mask.sum()),
    'm1_a': a, 'm1_b': b, 'train_mean': mu, 'train_median': med,
    'split': 'chronological_first_80pct_by_timestamp',
}, open(OUTPUT_DIR / 'b1_holdout_params.json', 'w'), indent=2)
display(table)

## Walk-forward (test set changes with k)

In [ ]:
wf_rows = []
for k in [k for k in K_VALUES if k < n - MIN_TEST_SIZE]:
    ev = evaluate_split(user_df.iloc[:k], user_df.iloc[k:])
    for method, res in ev.items():
        wf_rows.append({
            'k': k, 'n_train': k, 'n_test': n - k, 'method': method,
            'r2': res['r2'], 'mae': res['mae'], 'spearman': res['spearman'],
            'note': 'test_set_changes_with_k',
        })
wf = pd.DataFrame(wf_rows)
wf.to_csv(OUTPUT_DIR / 'b1_walkforward_curves.csv', index=False)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))
for ax, col, ylabel in zip(axes, ['r2', 'mae', 'spearman'], ['$R^2$', 'MAE', r'Spearman $\rho$']):
    for method, style in [('M0', '-'), ('M1', '-'), ('Naive_mean', '--'), ('Naive_median', ':')]:
        sub = wf[wf.method == method].sort_values('k')
        ax.plot(sub.k, sub[col], style, marker='o', markersize=3.5, label=method, lw=1.4)
    ax.set_xlabel('$k$ (train); test = remainder')
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3)
axes[0].legend(frameon=False, fontsize=8)
fig.suptitle('Walk-forward: test set changes with $k$', fontsize=10)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'fig1_learning_curves.png', dpi=150)
plt.show()

## Rolling-origin (interpreted summary)

In [ ]:
def block_bootstrap_mean_diff(err_a, err_b, block=6, n_boot=5000, seed=RANDOM_SEED):
    d = np.asarray(err_a, float) - np.asarray(err_b, float)
    n_ = len(d)
    if n_ < block + 1:
        return float(d.mean()), np.nan, np.nan
    rng = np.random.default_rng(seed)
    n_blocks = int(np.ceil(n_ / block))
    means = []
    for _ in range(n_boot):
        starts = rng.integers(0, n_ - block + 1, size=n_blocks)
        sample = np.concatenate([d[s:s + block] for s in starts])[:n_]
        means.append(sample.mean())
    lo, hi = np.percentile(means, [2.5, 97.5])
    return float(d.mean()), float(lo), float(hi)

burn = 10
roll_preds = {m: [] for m in ['M0', 'Naive_mean', 'Naive_median', 'Previous_rating', 'M1']}
y_list = []
y_all, p_all = user_df['y10'].values, user_df['p'].values
for t in range(burn, n):
    y_tr, p_tr = y_all[:t], p_all[:t]
    y_list.append(y_all[t])
    roll_preds['M0'].append(float(m0_predict([p_all[t]])[0]))
    roll_preds['Naive_mean'].append(float(np.mean(y_tr)))
    roll_preds['Naive_median'].append(float(np.median(y_tr)))
    roll_preds['Previous_rating'].append(float(y_all[t - 1]))
    if t >= 3:
        aa, bb = fit_m1(p_tr, y_tr)
        roll_preds['M1'].append(float(predict_m1(aa, bb, [p_all[t]])[0]))
    else:
        roll_preds['M1'].append(float(np.mean(y_tr)))

y_te_roll = np.asarray(y_list, float)
roll_rows = []
for method, pred in roll_preds.items():
    m = metrics_triple(y_te_roll, np.asarray(pred, float))
    roll_rows.append({'protocol': 'rolling_origin', 'burn_in': burn, 'method': method, **m})

err_m1 = np.abs(np.asarray(roll_preds['M1']) - y_te_roll)
err_mean = np.abs(np.asarray(roll_preds['Naive_mean']) - y_te_roll)
diff, lo, hi = block_bootstrap_mean_diff(err_m1, err_mean)
roll_rows.append({
    'protocol': 'rolling_origin_paired',
    'burn_in': burn,
    'method': 'M1_minus_Naive_mean_abs_err',
    'r2': np.nan, 'mae': diff, 'spearman': np.nan,
    'ci95_lo': lo, 'ci95_hi': hi,
    'note': 'moving_block_bootstrap_paired_diff',
})
roll = pd.DataFrame(roll_rows)
roll.to_csv(OUTPUT_DIR / 'b1_rolling_origin.csv', index=False)
display(roll)

## EDA figure

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.3))
axes[0].scatter(user_df['p'], user_df['y10'], alpha=0.7, s=22, edgecolor='k', linewidth=0.25)
axes[0].set_xlabel('$p$'); axes[0].set_ylabel('Self-report')
axes[1].hist(user_df['y10'], bins=np.arange(0.5, 11.5, 1), edgecolor='k', linewidth=0.4)
axes[1].set_xlabel('Rating'); axes[1].set_ylabel('Count')
vc = user_df['activity_type_norm'].value_counts()
axes[2].bar(vc.index.astype(str), vc.values, edgecolor='k', linewidth=0.4)
axes[2].set_xlabel('activity_type (descriptive)')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'eda_overview.png', dpi=150)
plt.show()
print('Wrote', OUTPUT_DIR)